# Parselmouth: Praat in Python

This is a brief companion notebook to Lecture 12. Everything we did in
Praat's GUI — loading audio, viewing spectrograms, measuring formants —
can also be done programmatically in Python using
[**Parselmouth**](https://parselmouth.readthedocs.io/), a Python
interface to Praat's analysis engine.

**Why would you want this?**
- Automate measurements across hundreds of files (no clicking)
- Integrate acoustic analysis into a Python data pipeline
- Produce reproducible analyses (a script is documentation of what you did)

Parselmouth is **not** part of our course environment. This notebook is
optional — it's here for students who want a Python-based workflow for
speech projects.

**To install** (in a terminal, with your `ling250` environment active):

```bash
pip install praat-parselmouth
```

In [ ]:
import parselmouth
from parselmouth.praat import call
import numpy as np
import matplotlib.pyplot as plt

---

## Loading Audio

Parselmouth uses the same `Sound` object as Praat internally.

In [ ]:
sound = parselmouth.Sound('../audio/kdt_001.wav')

print(f"Duration: {sound.duration:.3f} seconds")
print(f"Sample rate: {int(sound.sampling_frequency)} Hz")
print(f"Number of samples: {sound.n_samples:,}")

---

## Spectrogram

Parselmouth computes spectrograms using the same algorithm as Praat's
editor. The result is a Praat `Spectrogram` object, which we can convert
to a numpy array for plotting.

In [ ]:
spectrogram = sound.to_spectrogram()

# Convert to a numpy array (power in Pa^2/Hz) and take 10*log10 for dB
spec_data = spectrogram.values  # (n_freq_bins, n_time_frames)
spec_db = 10 * np.log10(spec_data + 1e-20)

plt.figure(figsize=(12, 5))
plt.imshow(
    spec_db,
    origin='lower',
    aspect='auto',
    extent=[
        spectrogram.xmin, spectrogram.xmax,
        spectrogram.ymin, spectrogram.ymax,
    ],
    cmap='afmhot',
)
plt.ylim(0, 5000)
plt.xlabel('Time (s)')
plt.ylabel('Frequency (Hz)')
plt.colorbar(label='Power (dB)')
plt.title('Spectrogram (via Parselmouth)')
plt.tight_layout()
plt.show()

---

## Pitch (F0) Tracking

Praat's pitch tracking algorithm is considered one of the best available.
With Parselmouth, it's one line.

In [ ]:
pitch = sound.to_pitch()

# Extract pitch values — unvoiced frames come back as 0
pitch_values = pitch.selected_array['frequency']
pitch_times = pitch.xs()

# Replace 0 (unvoiced) with NaN so they don't plot
pitch_values_clean = pitch_values.copy()
pitch_values_clean[pitch_values_clean == 0] = np.nan

plt.figure(figsize=(12, 3))
plt.plot(pitch_times, pitch_values_clean, 'o', markersize=3)
plt.xlabel('Time (s)')
plt.ylabel('F0 (Hz)')
plt.title('Pitch contour')
plt.tight_layout()
plt.show()

---

## Formant Extraction

This is the Python equivalent of clicking "Show formants" in Praat's
editor. We extract a `Formant` object and then query it for F1 and F2
at specific times.

In [ ]:
formants = sound.to_formant_burg()

# Extract F1 and F2 at every analysis frame
times = formants.xs()
f1_values = [formants.get_value_at_time(1, t) for t in times]
f2_values = [formants.get_value_at_time(2, t) for t in times]

fig, axes = plt.subplots(2, 1, figsize=(12, 5), sharex=True)

axes[0].plot(times, f1_values, 'o', markersize=2, label='F1')
axes[0].set_ylabel('Frequency (Hz)')
axes[0].set_title('F1 over time')
axes[0].set_ylim(0, 1200)

axes[1].plot(times, f2_values, 'o', markersize=2, color='C1', label='F2')
axes[1].set_ylabel('Frequency (Hz)')
axes[1].set_xlabel('Time (s)')
axes[1].set_title('F2 over time')
axes[1].set_ylim(500, 3000)

plt.tight_layout()
plt.show()

### Measuring formants at a specific time

Instead of clicking in Praat's editor and using "Get first formant,"
we can query any time point directly:

In [ ]:
# Pick a time in the middle of a vowel (adjust this for your recording)
measurement_time = 0.35

f1 = formants.get_value_at_time(1, measurement_time)
f2 = formants.get_value_at_time(2, measurement_time)

print(f"At t = {measurement_time:.2f} s:")
print(f"  F1 = {f1:.0f} Hz")
print(f"  F2 = {f2:.0f} Hz")

---

## Batch Measurement Example

The real power of Parselmouth is doing this at scale. Suppose you have a
list of vowel midpoints (from a TextGrid or a spreadsheet) — you can
extract formants from all of them in a loop and collect the results in a
table.

Here's the pattern, using made-up times as a placeholder:

In [ ]:
import pandas as pd

# In practice, these would come from a TextGrid or annotation file
vowel_data = [
    {'label': 'vowel_1', 'midpoint': 0.20},
    {'label': 'vowel_2', 'midpoint': 0.35},
    {'label': 'vowel_3', 'midpoint': 0.55},
]

results = []
for vowel in vowel_data:
    t = vowel['midpoint']
    results.append({
        'label': vowel['label'],
        'time': t,
        'F1': formants.get_value_at_time(1, t),
        'F2': formants.get_value_at_time(2, t),
    })

df = pd.DataFrame(results)
print(df.to_string(index=False))

---

## Reading TextGrids

Parselmouth can also read Praat TextGrid files. This means you can
annotate in Praat's GUI (where it's easiest) and then extract
measurements in Python (where batch processing is easiest).

In [ ]:
# Uncomment and adjust the path if you saved a TextGrid in the Praat exercise:
#
# textgrid = parselmouth.read('../audio/kdt_001.TextGrid')
#
# # List tier names
# for i in range(call(textgrid, 'Get number of tiers')):
#     tier_name = call(textgrid, 'Get tier name', i + 1)
#     print(f"Tier {i + 1}: {tier_name}")
#
# # Get all intervals from a tier
# tier_num = 1  # adjust as needed
# n_intervals = call(textgrid, 'Get number of intervals', tier_num)
# for j in range(1, n_intervals + 1):
#     label = call(textgrid, 'Get label of interval', tier_num, j)
#     start = call(textgrid, 'Get start time of interval', tier_num, j)
#     end = call(textgrid, 'Get end time of interval', tier_num, j)
#     if label:  # skip empty intervals
#         print(f"  [{start:.3f} - {end:.3f}] {label}")

---

## The `call()` Escape Hatch

Parselmouth wraps common Praat operations as Python methods (like
`.to_pitch()`, `.to_formant_burg()`). But if you need a Praat function
that doesn't have a dedicated Python wrapper, you can use `call()` to
invoke any Praat command by name:

```python
# Equivalent to Praat's "Get minimum pitch"
min_pitch = call(pitch, 'Get minimum', 0, 0, 'Hertz', 'Parabolic')
```

The arguments to `call()` match the Praat scripting interface. If you
know how to do something in Praat's scripting language, you can
translate it to Parselmouth's `call()` almost directly. See the
[Parselmouth docs](https://parselmouth.readthedocs.io/) for details.

---

## Summary

| Task | Praat GUI | Parselmouth |
|------|-----------|-------------|
| Load audio | Open → Read from file | `parselmouth.Sound(path)` |
| Spectrogram | View & Edit (automatic) | `sound.to_spectrogram()` |
| Pitch tracking | Pitch → Show pitch | `sound.to_pitch()` |
| Formant tracking | Formant → Show formants | `sound.to_formant_burg()` |
| Get F1 at time | Formant → Get first formant | `formants.get_value_at_time(1, t)` |
| Read TextGrid | Open → Read from file | `parselmouth.read(path)` |
| Any Praat command | Menus / scripting | `call(object, 'Command', ...)` |

**Use Praat** when you want to explore, annotate, and listen interactively.
**Use Parselmouth** when you want to extract measurements from many files
or integrate acoustic analysis into a Python pipeline.

For more: [parselmouth.readthedocs.io](https://parselmouth.readthedocs.io/)